# 章节实践

## 概述

通过本章的学习，你已经掌握了Vector算子的计算模型（3.2）、数据切分策略（3.3），以及Vector算子设计→实现→编译→验证的完整流程（3.4，并给出了Add算子的完整实现代码）。本节是课后实践，请参照3.4的开发流程，独立完成ReLU算子的开发、编译运行与验证，遇到困难时可回看3.4的Add算子实现，或对照本节末尾的参考答案。

In [ ]:
# 环境初始化
import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))
print("环境初始化完成")

---
# ReLU算子开发

Add是双目算子，ReLU是单目算子。请参照3.4介绍的Add算子开发流程与完整实现代码，独立完成ReLU算子的开发。

**算子要求：**

- 数学表达式：`z = max(x, 0)`（逐元素取正，单目算子）
- 单个输入`x`，输出`z`；dtype float32，shape `(8, 2048)`，format ND
- 核函数命名`vrelu_kernel`
- 主要接口：`asc.data_copy`、`asc.relu`

下面的脚手架只提供导入、常量与程序入口。**核函数区域与`vrelu_launch`都需要你自己编写**，写入`answer/03.05_chapter_practice/user/relu_framework.py`：

In [ ]:
%%writefile answer/03.05_chapter_practice/user/relu_framework.py
# Copyright (c) 2025 Huawei Technologies Co., Ltd.

import logging
import argparse
import torch
try:
    import torch_npu
except ModuleNotFoundError:
    pass

import asc
import asc.runtime.config as config
import asc.lib.runtime as rt

BUFFER_NUM = 2
USE_CORE_NUM = 8
TILE_NUM = 8

logging.basicConfig(level=logging.INFO)


# ==================== TODO：请编写ReLU核函数与Launch函数 ====================
# 需要自行实现：
#   - vrelu_kernel 及三段式 Device 函数（CopyIn -> Compute(asc.relu) -> CopyOut）
#   - vrelu_launch(x)：准备输出、计算 block_length / tile_length、调用核函数并返回结果



# ========================= TODO 区域结束 =========================


def vrelu_custom(backend: config.Backend, platform: config.Platform):
    config.set_platform(backend, platform)
    device = "npu" if config.Backend(backend) == config.Backend.NPU else "cpu"
    size = 8 * 2048
    x = torch.rand(size, dtype=torch.float32, device=device) - 0.5
    z = vrelu_launch(x)
    assert torch.allclose(z, torch.relu(x))


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("-r", type=str, default="NPU", help="backend to run")
    parser.add_argument("-v", type=str, default=None, help="platform to run")
    args = parser.parse_args()
    backend = args.r
    platform = args.v
    if backend not in config.Backend.__members__:
        raise ValueError("Unsupported Backend! Supported: ['Model', 'NPU']")
    backend = config.Backend(backend)
    if platform is not None:
        platform_values = [platform.value for platform in config.Platform]
        if platform not in platform_values:
            raise ValueError(f"Unsupported Platform! Supported: {platform_values}")
        platform = config.Platform(platform)
    logging.info("[INFO] start process sample relu_framework.")
    vrelu_custom(backend, platform)
    logging.info("[INFO] Sample relu_framework run success.")

编写完成后，执行以下命令编译运行你的ReLU算子。看到`Sample relu_framework run success.`即表示开发成功：

In [ ]:
!mkdir -p answer/03.05_chapter_practice/user
!python3 answer/03.05_chapter_practice/user/relu_framework.py -r NPU

---
## 参考答案

如遇困难，可执行以下代码查看ReLU的参考实现：

In [ ]:
!cat answer/03.05_chapter_practice/reference/relu_framework.py